In [1]:

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor

sys.path.append('../')
sys.path.append('./')
import importlib
import yaml
import torch
from tqdm.auto import tqdm
import logging

# logging.getLogger('sox').setLevel(logging.ERROR)
# logger = logging.getLogger('sox')
# logger.setLevel('CRITICAL')


In [2]:
import lightning_scripts.lightning_ssl_matched_speech_in_noise as lightning 
importlib.reload(lightning)


LitAudioSSL = lightning.LitAudioSSL

## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 4
config['hparas']['batch_size'] = 64
config['hparas']['global_batch_size'] = 64
config['num_gpus'] = 1 

model = LitAudioSSL(config)


In [3]:
from lightning_scripts import jsinV3DataLoader_precombined_batched 

importlib.reload(jsinV3DataLoader_precombined_batched)
MatchedSpeechInNoiseDatasetBatched = jsinV3DataLoader_precombined_batched.MatchedSpeechInNoiseDatasetBatched

dataset = MatchedSpeechInNoiseDatasetBatched(speech_h5_path=config['data']['val_speech_h5_path'],
                                                     noise_h5_path=config['data']['val_noise_h5_path'],
                                                     low_db=config['audio_transforms']['low_snr'],
                                                     high_db=config['audio_transforms']['high_snr'],
                                                     db_spl=config['audio_transforms']['dbspl'],
                                                     batch_size=config['hparas']['batch_size'],
                                                     signal_augment=config['data'].get("signal_augment", False),
                                                     target_keys=config['data'].get("target_keys", None),
                                                     )
dataset[0]

([tensor([[ 3.5031e-03, -5.6195e-04, -8.7461e-04,  ..., -1.8616e-02,
            9.7532e-04,  1.8969e-02],
          [-3.4944e-02, -3.6272e-02, -3.1833e-02,  ...,  1.0038e-02,
            1.9834e-02,  2.2699e-02],
          [-3.1922e-02, -9.3119e-03,  9.7585e-05,  ...,  1.8067e-02,
            1.6281e-02,  1.4803e-02],
          ...,
          [ 5.3410e-03, -3.6165e-03, -3.9945e-03,  ..., -2.0694e-02,
           -2.1402e-02, -2.4301e-02],
          [-6.8759e-02, -6.6595e-02, -5.8851e-02,  ...,  9.3283e-03,
            8.5087e-03,  7.5759e-03],
          [ 8.1187e-04,  6.7163e-04,  1.8561e-03,  ..., -1.3067e-02,
           -9.9122e-03, -1.3029e-02]]),
  tensor([[ 0.0143,  0.0135,  0.0131,  ..., -0.0007, -0.0030,  0.0015],
          [ 0.0037, -0.0010,  0.0013,  ..., -0.0087, -0.0082, -0.0173],
          [ 0.0050,  0.0037,  0.0026,  ...,  0.0044, -0.0003, -0.0039],
          ...,
          [-0.0263, -0.0208, -0.0174,  ..., -0.0027,  0.0015,  0.0021],
          [-0.0056, -0.0023,  0.0059, 

In [4]:
trainer = L.Trainer(
                    # callbacks=[lr_monitor],
                    # limit_train_batches=5,
                    limit_val_batches=2,
                    max_epochs=5,
                    # callbacks=callbacks,
                    #  strategy='ddp_notebook',
                    #  reload_dataloaders_every_n_epochs=-1,
                    devices=1)

trainer.fit(model)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name            | Type                       | Params | Mode 
-----------------------------------------------------------------------
0 | audio_rep       | AudioToAudioRepresentation | 0      | train
1 | model           | ModelWithFrontEnd          | 116 M  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Rank 0 N training batches 317
This is the first step after restoring from a checkpoint!


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined